# 03 Prove behavior with evidence
## Learning objective
Replay the workflow, reconcile all seven evidence layers, and separate quality, security and human release decisions.
## Technical level
L300-L600. Expected duration: 20 minutes.
## Prerequisites and setup
Notebook 00; Python 3.12 `.venv`. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
Offline synthetic data; no Azure cost, identity or licensing. Cloud model judges are intentionally not run. Quality metrics here are task/tool/citation proxies, not semantic model judging.


In [ ]:
from copy import deepcopy
from pathlib import Path
from uuid import uuid4

from foundry_security_accelerator.contracts import EvaluationEvidence, EvaluatorVerdict, Outcome
from foundry_security_accelerator.evaluation import adjudicate, reconcile, release_gate
from foundry_security_accelerator.redteam import load_items, reconcile_items
from foundry_security_accelerator.workflow_evaluation import evaluate_workflow

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
directory = ROOT / ".fsa" / "notebooks" / str(uuid4())
report = evaluate_workflow(
    ROOT / "data/evaluation/workflow.jsonl", ROOT / "data/evaluation/thresholds.toml", directory
)
assert report["status"] == "PASS"
assert report["metrics"]["policy_violations"] == 0
print({"kind": report["kind"], "status": report["status"], "metrics": report["metrics"]})
payload = load_items(ROOT / "data/redteam/synthetic_items.json")
before = deepcopy(payload.model_dump())
redteam = reconcile_items(payload)
assert payload.model_dump() == before
assert redteam["strict_failed"] is True
assert redteam["summary"]["outcome_counts"]["UNKNOWN"] == 1
print(redteam["summary"])

In [ ]:
conflict = EvaluationEvidence(
    case_id="synthetic-conflict",
    verdicts=(
        EvaluatorVerdict(evaluator="a", outcome=Outcome.PASS),
        EvaluatorVerdict(evaluator="b", outcome=Outcome.FAIL),
    ),
)
assert reconcile(conflict).outcome == Outcome.REVIEW_REQUIRED
missing = EvaluationEvidence(case_id="synthetic-missing", verdicts=())
assert release_gate((reconcile(missing),)).outcome == Outcome.INSUFFICIENT_EVIDENCE
assert release_gate((reconcile(conflict),)).outcome != Outcome.PASS
human = adjudicate(
    conflict,
    "synthetic-reviewer",
    "Synthetic exercise only; conflicting evidence remains unresolved.",
    Outcome.REVIEW_REQUIRED,
    directory / "human-adjudication.jsonl",
)
assert human.outcome == Outcome.REVIEW_REQUIRED
print(
    {
        "conflict": reconcile(conflict).outcome.value,
        "missing": reconcile(missing).outcome.value,
        "human_record": "SEPARATE_NOT_RELEASE_APPROVAL",
    }
)

## Today's presentation safe/unsafe gate
These package-backed checks measure the presentation's original synthetic local cases. The unsafe branch is deliberately vulnerable in-memory simulation only; it cannot enable a remote tool, cloud export, approval bypass or production write. Use its measured output, not historical screenshot metrics.


In [ ]:
from foundry_security_accelerator.presentation_runtime import evaluate_presentation

safe = evaluate_presentation(directory / "safe-presentation")
unsafe = evaluate_presentation(directory / "unsafe-presentation", unsafe=True)
assert safe["status"] == "PASS"
assert unsafe["status"] == "FAIL"
assert unsafe["metrics"]["attack_success_rate"] > 0
print({"safe": safe["status"], "unsafe": unsafe["status"], "metrics": unsafe["metrics"]})

## Expected results and validation
Sixteen local cases meet their thresholds; the selected red-team sample still contains confirmed unsafe and UNKNOWN candidates, so strict reconciliation fails as expected. Run `uv run pytest -m 'security or evaluation'` for negative controls.
## Security explanation
Raw records are unchanged; human adjudication is separate and not automatic approval. Missing action evidence never counts as PASS. The synthetic sample is not a population-level attack-success estimate. A safe final answer cannot overrule unsafe execution. Strict red-team rejection is narrower than the general release gate; POLICY_REVIEW still needs human decision.
## Troubleshooting
Missing/invalid inputs or evidence: stop, retain raw records, review schema; never replace missing evidence with a safe default.
## Cleanup
Retain originals privately under `.fsa` while reviewing. Clear outputs before sharing. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[evaluation](../src/foundry_security_accelerator/evaluation.py), [workflow replay](../src/foundry_security_accelerator/workflow_evaluation.py), [red-team](../src/foundry_security_accelerator/redteam.py).
